# Assignment 4: Polyglot Parity

Python/pandas version of the grouped BMI summary in `polyglot-parity.qmd`. Run from the repository root.

In [9]:
from pathlib import Path
import sys

import pandas as pd

helper_dir = Path("assignments/assignment04-polyglot/submission")
if not helper_dir.exists():
    helper_dir = Path(".")
sys.path.insert(0, str(helper_dir.resolve()))

from helpers import summarize_bmi_by_group

In [3]:
data_path = Path("examples/nhanes-equity/data/nhanes_equity_v6.csv")
if not data_path.exists():
    data_path = Path("../../../") / data_path

nhanes = pd.read_csv(data_path)
nhanes.info()

<class 'pandas.DataFrame'>
RangeIndex: 105626 entries, 0 to 105625
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Cycle           105626 non-null  str    
 1   BMI             96270 non-null   float64
 2   Weight          103936 non-null  float64
 3   Height          96738 non-null   float64
 4   Waist           92889 non-null   float64
 5   WeightMEC       105626 non-null  float64
 6   Strata          105626 non-null  int64  
 7   PSU             105626 non-null  int64  
 8   Gender          105626 non-null  str    
 9   Race            105626 non-null  str    
 10  Education       58375 non-null   str    
 11  Marital         58665 non-null   str    
 12  Age             105626 non-null  int64  
 13  PIR             96211 non-null   float64
 14  EducationClean  58375 non-null   str    
 15  MaritalClean    58665 non-null   str    
 16  IncomeGroup     96211 non-null   str    
 17  WHtR            92647

/tmp/ipykernel_12390/3886934413.py:5: DtypeWarning: Columns (11: Marital, 15: MaritalClean) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv(data_path)


## Data Cleaning

Remove rows missing either grouping variable. Retain missing BMI values so they can be counted within each income/gender group.

In [6]:
r_summary = nhanes.dropna(subset=["IncomeGroup", "Gender"]).copy()
r_summary.head()

,Cycle,BMI,Weight,Height,Waist,WeightMEC,Strata,PSU,Gender,Race,Education,Marital,Age,PIR,EducationClean,MaritalClean,IncomeGroup,WHtR
0,1999-2000,14.90,12.5,91.6,45.7,10982.898896,5,1,Female,Non-Hispanic Black,NaN,NaN,2,0.86,NaN,NaN,Low Income (<1.3),0.498908
1,1999-2000,24.90,75.4,174.0,98.0,28325.384898,1,3,Male,Non-Hispanic White,College Grad,NaN,77,5.00,College Grad,NaN,High Income (>3.5),0.563218
2,1999-2000,17.63,32.9,136.6,64.7,46192.256945,7,2,Female,Non-Hispanic White,NaN,NaN,10,1.47,NaN,NaN,Middle Income,0.473646
3,1999-2000,NaN,13.3,NaN,NaN,10251.260020,2,1,Male,Non-Hispanic Black,NaN,NaN,1,0.57,NaN,NaN,Low Income (<1.3),NaN
4,1999-2000,29.10,92.5,178.3,99.9,99445.065735,8,2,Male,Non-Hispanic White,College Grad,Married/Partner,49,5.00,College Grad,Married/Partner,High Income (>3.5),0.560292


## Descriptive BMI Summary

The standard deviation uses the sample definition (`ddof=1`), and the interquartile range uses the 25th and 75th percentiles.

In [10]:
bmi_summary = summarize_bmi_by_group(nhanes)
bmi_summary

,IncomeGroup,Gender,n,n_missing_bmi,mean_bmi,sd_bmi,median_bmi,iqr_bmi
0,High Income (>3.5),Female,11743,721,25.74,7.51,24.60,9.40
1,High Income (>3.5),Male,12095,779,26.11,6.81,26.00,8.47
2,Low Income (<1.3),Female,16548,2078,25.95,8.65,24.66,11.93
3,Low Income (<1.3),Male,14847,2130,24.05,7.17,23.30,10.16
4,Middle Income,Female,16432,1405,26.39,8.20,25.30,10.79
5,Middle Income,Male,15982,1451,25.52,7.22,25.10,9.59


In [11]:
repo_root = Path(".")
if not (repo_root / "examples").exists():
    repo_root = Path("../../../")

output_dir = repo_root / "assignments/assignment04-polyglot/submission/output"
output_dir.mkdir(parents=True, exist_ok=True)

python_metrics = pd.DataFrame(
    {
        "metric": ["raw_row_count", "raw_column_count", "filtered_row_count", "number_of_groups"],
        "value": [len(nhanes), len(nhanes.columns), len(r_summary), len(bmi_summary)],
    }
)
python_metrics.to_csv(output_dir / "python_metrics.csv", index=False)
bmi_summary.to_csv(output_dir / "python_bmi_summary.csv", index=False)

python_metrics

,metric,value
0,raw_row_count,105626
1,raw_column_count,18
2,filtered_row_count,96211
3,number_of_groups,6
